# Chapter 7: Tool Integration

Estimated time: ~7 hours.

Prerequisites: Chapter 1 (`agentlib.llm_client`), Chapter 4 (retry with backoff).

## Concept: MCP, schema validation, and a failure taxonomy for tools

#### Why a protocol at all

Before the Model Context Protocol, every agent framework that wanted to call external tools
needed a bespoke integration per tool: an *M×N* problem, M agent frameworks each writing
custom glue for N tools. MCP (Anthropic, 2024) standardizes the interface instead. Any
MCP-compliant client can talk to any MCP-compliant server, the same way any HTTP client can
talk to any HTTP server, without either side knowing the other's internals in advance. A
server exposes a set of *tools* (each with a name, description, and a JSON Schema for its
inputs); a client discovers those tools at connection time and calls them by name with
structured arguments. This chapter builds one of each, a real server and a real client that
talks to it, not a description of the protocol.

#### Transport

This chapter uses stdio transport: the client spawns the server as a subprocess and the two
exchange JSON-RPC messages over the subprocess's stdin/stdout. This is MCP's simplest, most
common transport for local tools, with no network and no ports, just a pipe. (The protocol
also supports HTTP-based transports for remote servers; not needed here.)

#### Schema validation

A tool's declared input schema is a contract, not documentation. The client should validate
against it, and so should the server's own output shape if the caller has any expectations
about what comes back. This chapter uses Pydantic for that validation, the same library real
MCP server implementations widely use for exactly this purpose.

#### A failure taxonomy

Not every tool failure needs the same response, and treating them identically (retry
everything, or worse, silently swallow everything) is a common real mistake. Four distinct
failure shapes need four different responses:

| Failure type | What it looks like | What usually helps |
|---|---|---|
| **Transient** | A timeout, a connection reset, a rate limit: the request was fine, the environment glitched | Retry, with backoff |
| **Malformed** | The response doesn't parse or doesn't match its schema: a field has the wrong type, or is missing | Switch to an alternate source/path; retrying the *same* call rarely helps if the corruption is upstream |
| **Semantically wrong** | The response is well-formed and schema-valid, but doesn't actually answer what was asked (e.g., data for a different entity than requested) | Ask a human/verification step; retrying blindly can't tell "wrong" from "right" without ground truth |
| **Version mismatch** | The response shape itself has changed: a field renamed or restructured, suggesting the upstream API changed | Ask a developer to update the integration; this isn't a runtime-recoverable failure |

This chapter's break-it section builds a decision router that classifies a failure into one
of these four buckets and picks the right response (retry, switch, or ask-user) instead of a
single generic error handler.

## Setup

The MCP server this chapter builds and calls is real: a genuine tool exposing real package
metadata from PyPI's live JSON API (`https://pypi.org/pypi/<package>/json`), not a mock. The
build spec originally called for a filtered GH Archive slice as this chapter's real-data
source; GH Archive is Hugging-Face-hosted, and this build environment can't reach
`huggingface.co` (the same reachability issue noted for SQuAD in Chapter 3 and `tiktoken` in
Chapter 5). PyPI's JSON API needed no such workaround at all. It's directly reachable, live,
and arguably a better fit for demonstrating a real local MCP server anyway, since a tool that
looks up live external data is closer to how MCP servers get used in practice than a static
dataset dump would be. See `PROGRESS.md`'s Unit 8 notes for the full reasoning.

In [1]:
import json
import os
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from pydantic import BaseModel, ValidationError

from agentlib import llm_client
from agentlib.grading import check

_MCP_SERVER_PATH = str(_repo_root / "curriculum" / "_ch07_mcp_server.py")
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")
print(f"MCP server script: {_MCP_SERVER_PATH}")


LLM_PROVIDER = 'anthropic', HAS_KEY = False
MCP server script: /home/user/learning-agentic-ai/curriculum/_ch07_mcp_server.py


## Build: connect to the real server, validate its schema

`curriculum/_ch07_mcp_server.py` is a companion file, not notebook code: MCP's stdio
transport spawns the server as a genuine separate process, so it has to exist as a real
script the client can launch. Open that file alongside this notebook; it's short. Its one
tool, `get_package_info(package_name)`, reads from a small local cache of real PyPI data
(`data/pypi_cache/`, three real packages already fetched: `requests`, `numpy`, `anthropic`)
and falls back to a live PyPI query for anything not cached.

In [2]:
class PackageInfo(BaseModel):
    '''The schema contract for what this tool should return. Optional fields really are
    optional in real PyPI data -- e.g. many packages leave the legacy `home_page` field
    empty and only populate `project_urls` -- but name/version/summary should always be
    present for a real package.'''
    name: str
    version: str
    summary: str
    license: str | None = None
    home_page: str | None = None
    project_urls: dict | None = None


schema_path = _repo_root / "data" / "schemas" / "package_info.schema.json"
schema_path.write_text(json.dumps(PackageInfo.model_json_schema(), indent=2) + "\n")
print(f"Wrote schema to {schema_path.relative_to(_repo_root)}")
print(json.dumps(PackageInfo.model_json_schema(), indent=2))


Wrote schema to data/schemas/package_info.schema.json
{
  "description": "The schema contract for what this tool should return. Optional fields really are\noptional in real PyPI data -- e.g. many packages leave the legacy `home_page` field\nempty and only populate `project_urls` -- but name/version/summary should always be\npresent for a real package.",
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "version": {
      "title": "Version",
      "type": "string"
    },
    "summary": {
      "title": "Summary",
      "type": "string"
    },
    "license": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "License"
    },
    "home_page": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Home Page"
    },
    "project_urls": {
     

### Validating, without swallowing the failure

`model_validate` raises on bad data, so somewhere there has to be a `try`. What that `except`
block returns is one of the highest-leverage decisions in this chapter.

The tempting version catches `ValidationError` and returns `None`. It's tidy, it never
crashes, and it destroys information the caller needs: `None` from a lookup already means
"no such package". Collapse "the tool returned garbage" into that same value and a broken
integration becomes indistinguishable from an empty result, so the agent reports "I couldn't
find anything about numpy" with total confidence and nobody discovers the real problem for
weeks.

In [ ]:
def validate_tool_output(response: dict, model):
    '''Validate a raw tool response against its schema.

    Return a (validated, error) pair: (the parsed model, None) on success, or
    (None, the ValidationError) on failure. Exactly one of the two is ever populated.

    Return the ValidationError object itself, not a flattened string -- the caller needs
    .errors() to tell a field that is missing entirely from one that is present with the
    wrong type, and those two mean very different things about what broke upstream.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


validate_tool_output = check("ch07-schema-validate", validate_tool_output)

### The client itself

Four things have to happen in order, and the ordering is the part that bites: spawn the
server over stdio, open a session on its read/write streams, **complete the handshake**, and
only then call a tool.

MCP is stateful. `session.initialize()` is where client and server negotiate what the
connection can do, and a `call_tool` issued before that finishes has nothing to route. It
does not fail with a clear message; it hangs, or it comes back as a protocol error several
frames removed from the actual mistake.

In [ ]:
async def call_mcp_tool(server_path: str, tool_name: str, arguments: dict) -> dict:
    '''Spawn the real MCP server, call one of its tools, return the raw (unvalidated) result.

    Use StdioServerParameters(command=sys.executable, args=[server_path]) with stdio_client,
    open a ClientSession on the (read, write) pair, await session.initialize() BEFORE any
    call_tool, then return the parsed JSON payload as a dict -- MCP hands back content blocks
    whose .text is a JSON string, so callers should not have to json.loads() it themselves.

    Pass errlog=devnull to stdio_client. That works around an environment quirk rather than
    teaching anything: under some headless notebook runners the kernel's stdout/stderr have
    no real file descriptor, which the subprocess machinery needs for the child's stderr.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


call_mcp_tool = check("ch07-mcp-client", call_mcp_tool)

In [ ]:
async def call_get_package_info(package_name: str) -> dict:
    return await call_mcp_tool(_MCP_SERVER_PATH, "get_package_info", {"package_name": package_name})


raw_result = await call_get_package_info("requests")
print("Raw tool result:", raw_result)

validated = PackageInfo.model_validate(raw_result)
print(f"\nValidated: {validated.name} v{validated.version} -- {validated.summary!r}")

## Break it: four failure shapes, injected into copies of a real response

Starting from the real, validated `numpy` response fetched above, each scenario below
injects exactly one of this chapter's four failure types into a copy of it, the same
approach Chapter 5 used for its request-log bugs. The base data is genuinely real; each
failure shape is a deliberate, clearly-labeled injection, not a real failure that happened to
occur (real MCP servers don't fail on demand, which is exactly why previous chapters'
break-it sections use the same technique).

In [4]:
real_response = await call_get_package_info("numpy")
print("Real base response:", real_response)


Real base response: {'name': 'numpy', 'version': '2.5.2', 'summary': 'Fundamental package for array computing in Python', 'license': None, 'home_page': None, 'project_urls': {'documentation': 'https://numpy.org/doc/', 'download': 'https://pypi.org/project/numpy/#files', 'homepage': 'https://numpy.org', 'release notes': 'https://numpy.org/doc/stable/release', 'source': 'https://github.com/numpy/numpy', 'tracker': 'https://github.com/numpy/numpy/issues'}}


### 1. Transient: the call itself fails, not the data

In [5]:
class _FlakyCallCounter:
    '''Simulates a tool call that times out the first couple of attempts, then succeeds --
    the transient-failure shape from this chapter's concept table. The underlying call is
    still the real one; only whether it's allowed to complete is being controlled here.'''
    def __init__(self, fail_times: int):
        self.fail_times = fail_times
        self.attempts = 0

    async def call(self, package_name: str) -> dict:
        self.attempts += 1
        if self.attempts <= self.fail_times:
            raise TimeoutError(f"attempt {self.attempts}: simulated timeout")
        return await call_get_package_info(package_name)


async def call_with_retry(fn, *args, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return await fn(*args)
        except TimeoutError as e:
            if attempt == max_attempts:
                raise
            print(f"  attempt {attempt} failed ({e}), retrying...")
    raise RuntimeError("unreachable")


flaky = _FlakyCallCounter(fail_times=2)
result = await call_with_retry(flaky.call, "anthropic")
print(f"\nSucceeded on attempt {flaky.attempts}: {result['name']} v{result['version']}")


  attempt 1 failed (attempt 1: simulated timeout), retrying...
  attempt 2 failed (attempt 2: simulated timeout), retrying...



Succeeded on attempt 3: anthropic v0.121.0


### 2. Malformed: the response arrives, but doesn't type-check

A field has the wrong type: the tool responded, the shape is close, but something in it is
corrupt. Schema validation catches this; the fix is treating the same tool's next call with
suspicion (switch to a fresh, uncached read) rather than trusting the same corrupted source
again.

In [6]:
malformed_response = dict(real_response)
malformed_response["version"] = 2.5  # should be the string "2.5.2" -- a float slipped in

try:
    PackageInfo.model_validate(malformed_response)
    print("Validated cleanly -- this shouldn't happen for this scenario.")
except ValidationError as e:
    print("Schema validation caught it:")
    for err in e.errors():
        print(f"  field {err['loc']}: {err['type']} -- {err['msg']}")


Schema validation caught it:
  field ('version',): string_type -- Input should be a valid string


### 3. Semantically wrong: valid shape, wrong content

Every field type-checks. The response is simply about the wrong thing. No schema can catch
this, because nothing about the shape is broken. This needs a check against what was
actually asked for, not just against the schema.

In [7]:
wrong_content_response = dict(real_response)
wrong_content_response["name"] = "not-numpy"  # a different package's data, somehow

validated = PackageInfo.model_validate(wrong_content_response)  # passes schema validation fine
print(f"Schema validation: OK ({validated.name} v{validated.version})")
print(f"Requested 'numpy', got data for {validated.name!r} -- schema alone can't catch this.")


Schema validation: OK (not-numpy v2.5.2)
Requested 'numpy', got data for 'not-numpy' -- schema alone can't catch this.


### 4. Version mismatch: the shape itself changed

Not a type error on a present field: a field is missing entirely, the signature of an
upstream API that changed shape (a rename, a restructure) out from under code that still
expects the old one.

In [8]:
version_mismatch_response = dict(real_response)
version_mismatch_response["pkg_version"] = version_mismatch_response.pop("version")  # renamed upstream

try:
    PackageInfo.model_validate(version_mismatch_response)
    print("Validated cleanly -- this shouldn't happen for this scenario.")
except ValidationError as e:
    for err in e.errors():
        print(f"  field {err['loc']}: {err['type']} -- {err['msg']}")


  field ('version',): missing -- Field required


## The decision router

One function that classifies a raw tool response into this chapter's taxonomy and picks the
matching response (retry, switch, or ask-user) instead of a single generic
`except Exception: log and move on`, which is what actually causes the "the agent just
silently returned garbage" failure mode this chapter opened with.

The line that matters most in this taxonomy is the one between **malformed** and
**semantically wrong**, because it is the one a schema can't draw for you. Malformed data
failed to type-check, and a retry against a fresh source may well fix it. Semantically wrong
data type-checked perfectly and describes something other than what was asked for -- and will
come back identical every single time, no matter how many times you retry it. Same validator,
opposite correct response.

In [ ]:
def classify_failure(package_name: str, response: dict, model) -> dict:
    '''Sort a raw tool response into this chapter's four categories.

    Return {"category": ..., "action": ..., "detail": ...} using exactly these pairings:

      "ok"                 -> "proceed"    validated, and it describes what was asked for
      "malformed"          -> "switch"     a field failed type validation; retry a fresh source
      "version_mismatch"   -> "ask-user"   a field is missing ENTIRELY; the upstream shape moved
      "semantically_wrong" -> "ask-user"   valid, well-typed, and about the wrong package

    Use validate_tool_output above. Tell missing from mistyped via the error's .errors() --
    a "missing" type means version_mismatch, anything else means malformed. Put something
    specific in `detail`: whoever reads the log should not have to go and look.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


classify_failure = check("ch07-failure-classifier", classify_failure)

In [ ]:
scenarios = {
    "clean": real_response,
    "malformed": malformed_response,
    "semantically wrong": wrong_content_response,
    "version mismatch": version_mismatch_response,
}

for label, response in scenarios.items():
    verdict = classify_failure("numpy", response, PackageInfo)
    print(f"[{label:19s}] {verdict['category']:19s} -> {verdict['action']:9s} {verdict['detail']}")

## Interview drill

Answer each of these on your own before checking
`solutions/ch07_tool_integration_answers.md`. Nothing here is answered inline in this
notebook.

1. Definitional. Why does MCP standardize the *client-server* interface specifically,
rather than each agent framework just agreeing on a shared library? What does the M×N framing
actually buy you?

2. Cold diagnosis. A tool call in production returns data that passes your schema
validation completely, but a user reports the agent gave a confidently wrong answer. Walk
through how you'd investigate, and which of this chapter's four failure types you'd suspect
first.

3. Design judgment. A teammate proposes handling every tool failure the same way: catch
the exception, retry three times with backoff, then give up. What's wrong with this as a
universal policy, using this chapter's taxonomy to be specific about where it breaks down?

4. Judgment call. You're integrating a third-party tool whose response schema has changed
twice in the last year without a version number changing in the API itself. What would you
build into your integration to catch that class of failure automatically, rather than
finding out from a user bug report?

## Recap

This chapter covered: what MCP standardizes and why (the M×N integration problem), stdio
transport as MCP's simplest local-tool mechanism, and schema validation as an enforced
contract rather than documentation. Built a real local MCP server (`_ch07_mcp_server.py`)
exposing genuine live PyPI package data, and a real client that connects to it over stdio,
not a simulation of either side. The break-it section worked through this chapter's
four-part failure taxonomy (transient, malformed, semantically wrong, version mismatch),
each demonstrated as a real, distinguishable failure shape, unified into a single decision
router that picks retry, switch, or ask-user based on what actually went wrong, instead of
one generic catch-all.

Next: Chapter 8 steps back from any single technique to system design, a repeatable
framework for approaching an agent design problem from scratch, including a question this
whole course has been implicitly answering chapter by chapter: when not to use an agent at
all.